In [ ]:
import os,sys
import re
import numpy as np
import pandas as pd
import scanpy as sc
from scanpy import AnnData
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from scipy.sparse import csr_matrix
import squidpy as sq
import anndata as ad
import mudata
import harmonypy as hm

sc.set_figure_params(figsize=(4,4))
plt.rcParams['pdf.fonttype']=42
warnings.filterwarnings("ignore")
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
sc.settings.verbosity = 3

In [ ]:
control_path='/u/home/m/mbaig/project-cluo/cosmx/spatial-data/20250527_MB_basal_2/'
dataDir_spatial = '/u/home/m/mbaig/project-cluo/cosmx/analyses/20250527_MB_basal_2/processed_data/'

In [ ]:
#loading in the raw data matrix in chunks
#slide1

# Initialize an empty list to store chunks
chunks = []

# Define the chunk size
chunksize = 100000

# Read the CSV file in chunks
for chunk in pd.read_csv(control_path + 'MBbrain6Kbasalslide3redo/MBbrain6Kbasalslide3redo_exprMat_file.csv.gz', chunksize=chunksize):
    chunks.append(chunk)

# Concatenate all chunks into a single DataFrame
df1 = pd.concat(chunks, ignore_index=True)

# Display the DataFrame
print(df1.head())

In [ ]:
meta=pd.read_csv(control_path+'MBbrain6Kbasalslide3redo/MBbrain6Kbasalslide3redo_metadata_file.csv.gz')
meta=meta[meta.columns[3:]].set_index('cell')
meta

In [ ]:
df = df1

In [ ]:
# create row names for "cell" in same format as what's used in cell metadata file
# the new cell_ID has pattern as 'c_[slideID]_[cell_ID]', where slideID = 1 in this example dataset
df["new_cell_ID"] =  df.apply(lambda row: f"c_1_{row.fov}_{row.cell_ID}", axis = 1)
df.set_index("new_cell_ID", inplace=True)

# drop annotation columns
df.drop(columns=["fov", "cell_ID"], inplace=True)

# focus on RNA probes while drop control probes
dummy=re.compile(r'Negative|SystemControl', flags=re.IGNORECASE)
chosen_probes = [col for col in list(df.columns) if not dummy.search(col)]

# "cell" names for ordering downstream
chosen_cells = df.index

# convert to a sparse matrix
raw = csr_matrix(df[chosen_probes].astype(pd.SparseDtype("float64",0)).sparse.to_coo())

# delete unused variable to free up memory
del df

In [ ]:
# read in cell meta data file
#cell_meta = pd.read_csv(os.path.join(dataDir, 'Pancreas_metadata_file.csv'))
cell_meta=pd.read_csv(control_path+'MBbrain6Kbasalslide3redo/MBbrain6Kbasalslide3redo_metadata_file.csv.gz')

# use "cell" column as row names to match with raw expression matrix
# for convenience, the "cell" column is kept with the meta data
cell_meta.set_index("cell", inplace=True, drop = False)

# extract spatial coordinates of cells, use the global coordinates
coords = cell_meta[["CenterX_global_px","CenterY_global_px"]]

# use "x" and "y" as column names
coords = coords.rename(columns={"CenterX_global_px":"x",
                                "CenterY_global_px": "y"})

# reorder cells to be in same row order as raw expression matrix
coords = coords.reindex(index = chosen_cells)

# convert px coordinates to micrometer
pixel_size = 0.12028
coords = coords.mul(pixel_size)

In [ ]:
adata = ad.AnnData(
    X = raw,
    obs = cell_meta,
    # row names should be the same as gene names
    var = pd.DataFrame(
        list(chosen_probes),
        columns = ["gene"],
        index = chosen_probes))

# add spatial coordinates
adata.obsm['spatial'] = coords.to_numpy()

# you can add name of slide or original file as one of unstructured annotation
adata.uns['name'] = "MBbrain6Kbasalslide3redo"

# convert string columns to categorical data in `obs`
adata.strings_to_categoricals()

In [ ]:
plt.rcParams['figure.figsize']=(10,14)
sns.scatterplot(data=adata.obs,x='CenterX_global_px', y='CenterY_global_px', s=2, edgecolor=None)  # s parameter sets dot size, edgecolor=None removes outline
plt.show()

In [ ]:
adata

In [ ]:
import numpy as np

# Specify the gene(s) of interest
gene_of_interest = 'STMN2'  # Replace with the name of the gene

# Ensure the gene is in the variable names
if gene_of_interest in adata.var.index:
    # Extract counts for the specific gene
    gene_counts = adata[:, gene_of_interest].X

    # Calculate the average count
    average_count = np.mean(gene_counts)

    print(f"The average count for {gene_of_interest} is {average_count}")
else:
    print(f"The gene {gene_of_interest} is not present in the dataset.")

In [ ]:
dataDir_spatial = '/u/home/m/mbaig/project-cluo/cosmx/analyses/20250527_MB_basal_2/processed_data/20250527'

In [ ]:
adata.write(os.path.join(dataDir_spatial, "MBbrain6Kbasalslide3redo_raw1.h5ad"), compression="gzip")

In [ ]:
#loading in the raw data matrix in chunks
#slide2

# Initialize an empty list to store chunks
chunks = []

# Define the chunk size
chunksize = 100000

# Read the CSV file in chunks
for chunk in pd.read_csv(control_path + 'MBbrain6Kbasalslide4redo/MBbrain6Kbasalslide4redo_exprMat_file.csv.gz', chunksize=chunksize):
    chunks.append(chunk)

# Concatenate all chunks into a single DataFrame
df1 = pd.concat(chunks, ignore_index=True)

# Display the DataFrame
print(df1.head())

In [ ]:
meta=pd.read_csv(control_path+'MBbrain6Kbasalslide4redo/MBbrain6Kbasalslide4redo_metadata_file.csv.gz')
meta=meta[meta.columns[3:]].set_index('cell')
meta

In [ ]:
# read in cell meta data file
#cell_meta = pd.read_csv(os.path.join(dataDir, 'Pancreas_metadata_file.csv'))
cell_meta=pd.read_csv(control_path+'MBbrain6Kbasalslide4redo/MBbrain6Kbasalslide4redo_metadata_file.csv.gz')

# use "cell" column as row names to match with raw expression matrix
# for convenience, the "cell" column is kept with the meta data
cell_meta.set_index("cell", inplace=True, drop = False)

# extract spatial coordinates of cells, use the global coordinates
coords = cell_meta[["CenterX_global_px","CenterY_global_px"]]

# use "x" and "y" as column names
coords = coords.rename(columns={"CenterX_global_px":"x",
                                "CenterY_global_px": "y"})

# reorder cells to be in same row order as raw expression matrix
coords = coords.reindex(index = chosen_cells)

# convert px coordinates to micrometer
pixel_size = 0.12028
coords = coords.mul(pixel_size)

In [ ]:
adata = ad.AnnData(
    X = raw,
    obs = cell_meta,
    # row names should be the same as gene names
    var = pd.DataFrame(
        list(chosen_probes),
        columns = ["gene"],
        index = chosen_probes))

# add spatial coordinates
adata.obsm['spatial'] = coords.to_numpy()

# you can add name of slide or original file as one of unstructured annotation
adata.uns['name'] = "MBbrain6Kbasalslide4redo"

# convert string columns to categorical data in `obs`
adata.strings_to_categoricals()

In [ ]:
adata.write(os.path.join(dataDir_spatial, "MBbrain6Kbasalslide4redo_raw1.h5ad"), compression="gzip")

In [ ]:
#slide4 now, still called adata1
adata1 = sc.read(os.path.join(dataDir_spatial, "20250527/MBbrain6Kbasalslide4redo_raw1.h5ad"))
adata1

In [ ]:
plt.rcParams['figure.figsize']=(10,14)
sns.scatterplot(data=adata1.obs,x='CenterX_global_px', y='CenterY_global_px', s=2, edgecolor=None)  # s parameter sets dot size, edgecolor=None removes outline
plt.show()

In [ ]:
#no filtering for now

adata1.layers["counts"] = adata1.X.copy()
sc.pp.normalize_total(adata1, inplace=True)
sc.pp.log1p(adata1)
sc.pp.pca(adata1)
sc.pp.neighbors(adata1)
sc.tl.umap(adata1)
sc.tl.leiden(adata1)

In [ ]:
sc.pl.umap(adata1, color=['leiden'], size=2)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set the figure size
plt.rcParams['figure.figsize'] = (10, 14)

# Create the scatter plot
sns.scatterplot(
    data=adata1.obs,
    x='CenterX_global_px',
    y='CenterY_global_px',
    hue='leiden',  # Color by Leiden clusters
    palette='tab20',  # Choose a color palette
    s=2,  # Set dot size
    edgecolor=None  # Remove outline
)

# Show the plot
plt.show()

In [ ]:
#now plotting back on spatial image

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Extract spatial and UMAP coordinates
spatial_coords = adata1.obs[['CenterX_global_px', 'CenterY_global_px']]
umap_coords = adata1.obsm['X_umap']
leiden_clusters = adata1.obs['leiden'].astype(str)  # Convert Leiden clusters to string for mapping

# Convert Leiden clusters to integers for proper numerical sorting
leiden_clusters_int = leiden_clusters.astype(int)
sorted_clusters = np.sort(leiden_clusters_int.unique())  # Sort numerically

# Define a consistent color palette
palette = sns.color_palette('tab20', len(sorted_clusters))
cluster_colors = {str(cluster): palette[i] for i, cluster in enumerate(sorted_clusters)}

# Create figure and subplots
fig, axes = plt.subplots(1, 2, figsize=(18, 9), gridspec_kw={'width_ratios': [1, 1]})

# Spatial plot
sns.scatterplot(x=spatial_coords['CenterX_global_px'], y=spatial_coords['CenterY_global_px'],
                hue=leiden_clusters, palette=cluster_colors, s=2, edgecolor=None, ax=axes[0])
axes[0].set_title('Leiden Clusters on Spatial Image')
axes[0].set_xlabel('CenterX (px)')
axes[0].set_ylabel('CenterY (px)')
axes[0].legend_.remove()  # Remove individual legend

# UMAP plot
sns.scatterplot(x=umap_coords[:, 0], y=umap_coords[:, 1],
                hue=leiden_clusters, palette=cluster_colors, s=5, edgecolor=None, ax=axes[1])
axes[1].set_title('Leiden Clusters on UMAP')
axes[1].set_xlabel('UMAP1')
axes[1].set_ylabel('UMAP2')
axes[1].legend_.remove()  # Remove individual legend

# Create a single shared legend closer to the plots
handles = [plt.Line2D([0], [0], marker='o', color=cluster_colors[str(cluster)], linestyle='', markersize=8)
           for cluster in sorted_clusters]
labels = [str(cluster) for cluster in sorted_clusters]
fig.legend(handles, labels, title="Leiden Clusters", bbox_to_anchor=(1, 0.5), loc='center left')

plt.tight_layout(rect=[0, 0, 0.85, 1])  # Adjust layout to fit legend closer to the plots
plt.show()

In [ ]:
dataDir_spatial_20250527 = '/u/home/m/mbaig/project-cluo/cosmx/analyses/20250527_MB_basal_2/processed_data/20250527/'

In [ ]:
adata1.write(os.path.join(dataDir_spatial_20250527, "MBbrain6Kbasalslide4redo_unfiltered_normalized1.h5ad"), compression="gzip")

In [ ]:
control_path='/u/home/m/mbaig/project-cluo/cosmx/spatial-data/20250527_MB_basal_2/MBbrain6Kbasalslide4redo - resegmented - 20260106/'
dataDir_spatial = '/u/home/m/mbaig/project-cluo/cosmx/analyses/20250527_MB_basal_2/processed_data/'

In [ ]:
#loading in the raw data matrix in chunks
#slide1

# Initialize an empty list to store chunks
chunks = []

# Define the chunk size
chunksize = 100000

# Read the CSV file in chunks
for chunk in pd.read_csv(control_path + 'MBbrain6Kbasalslide4redo_exprMat_file.csv.gz', chunksize=chunksize):
    chunks.append(chunk)

# Concatenate all chunks into a single DataFrame
df1 = pd.concat(chunks, ignore_index=True)

# Display the DataFrame
print(df1.head())

In [ ]:
meta=pd.read_csv(control_path+'MBbrain6Kbasalslide4redo_metadata_file.csv.gz')
meta=meta[meta.columns[3:]].set_index('cell')
meta

In [ ]:
# read in cell meta data file
#cell_meta = pd.read_csv(os.path.join(dataDir, 'Pancreas_metadata_file.csv'))
cell_meta=pd.read_csv(control_path+'MBbrain6Kbasalslide4redo_metadata_file.csv.gz')

# use "cell" column as row names to match with raw expression matrix
# for convenience, the "cell" column is kept with the meta data
cell_meta.set_index("cell", inplace=True, drop = False)

# extract spatial coordinates of cells, use the global coordinates
coords = cell_meta[["CenterX_global_px","CenterY_global_px"]]

# use "x" and "y" as column names
coords = coords.rename(columns={"CenterX_global_px":"x",
                                "CenterY_global_px": "y"})

# reorder cells to be in same row order as raw expression matrix
coords = coords.reindex(index = chosen_cells)

# convert px coordinates to micrometer
pixel_size = 0.12028
coords = coords.mul(pixel_size)

In [ ]:
adata = ad.AnnData(
    X = raw,
    obs = cell_meta,
    # row names should be the same as gene names
    var = pd.DataFrame(
        list(chosen_probes),
        columns = ["gene"],
        index = chosen_probes))

# add spatial coordinates
adata.obsm['spatial'] = coords.to_numpy()

# you can add name of slide or original file as one of unstructured annotation
adata.uns['name'] = "MBbrain6Kbasalslide4redo_reseg"

# convert string columns to categorical data in `obs`
adata.strings_to_categoricals()

In [ ]:
plt.rcParams['figure.figsize']=(10,14)
sns.scatterplot(data=adata.obs,x='CenterX_global_px', y='CenterY_global_px', s=1.5, edgecolor=None)  # s parameter sets dot size, edgecolor=None removes outline
plt.show()

In [ ]:
dataDir_spatial = '/u/home/m/mbaig/project-cluo/cosmx/analyses/20250527_MB_basal_2/processed_data/20260106'

In [ ]:
adata.write(os.path.join(dataDir_spatial, "MBbrain6Kbasalslide4_reseg_raw1.h5ad"), compression="gzip")

In [ ]:
adata1 = adata.copy()
del adata
adata1

In [ ]:
umi_counts = adata1.obs['nCount_RNA']

In [ ]:
umi_counts.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])

In [ ]:
adata1_filtered = adata1[(adata1.obs['nCount_RNA'] > 100) & (adata1.obs['nCount_RNA'] < 4000)].copy()
adata1_filtered

In [ ]:
plt.rcParams['figure.figsize']=(10,10)
sns.scatterplot(data=adata1_filtered.obs,x='CenterX_global_px', y='CenterY_global_px', s=1, edgecolor=None)  # s parameter sets dot size, edgecolor=None removes outline
plt.show()

In [ ]:
adata1_filtered

In [ ]:
#200 umi cutoff as a test

adata1_filtered_2 = adata1[(adata1.obs['nCount_RNA'] > 200) & (adata1.obs['nCount_RNA'] < 4000)].copy()
adata1_filtered_2

In [ ]:
plt.rcParams['figure.figsize']=(10,10)
sns.scatterplot(data=adata1_filtered_2.obs,x='CenterX_global_px', y='CenterY_global_px', s=1, edgecolor=None)  # s parameter sets dot size, edgecolor=None removes outline
plt.show()

In [ ]:
adata1_filtered_2

In [ ]:
adata1_filtered.layers["counts"] = adata1_filtered.X.copy()
sc.pp.normalize_total(adata1_filtered, inplace=True)
sc.pp.log1p(adata1_filtered)
sc.pp.pca(adata1_filtered)
sc.pp.neighbors(adata1_filtered)
sc.tl.umap(adata1_filtered)
sc.tl.leiden(adata1_filtered)

In [ ]:
#now plotting back on spatial image

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Extract spatial and UMAP coordinates
spatial_coords = adata1_filtered.obs[['CenterX_global_px', 'CenterY_global_px']]
umap_coords = adata1_filtered.obsm['X_umap']
leiden_clusters = adata1_filtered.obs['leiden'].astype(str)  # Convert Leiden clusters to string for mapping

# Convert Leiden clusters to integers for proper numerical sorting
leiden_clusters_int = leiden_clusters.astype(int)
sorted_clusters = np.sort(leiden_clusters_int.unique())  # Sort numerically

# Define a consistent color palette
palette = sns.color_palette('tab20', len(sorted_clusters))
cluster_colors = {str(cluster): palette[i] for i, cluster in enumerate(sorted_clusters)}

# Create figure and subplots
fig, axes = plt.subplots(1, 2, figsize=(18, 9), gridspec_kw={'width_ratios': [1, 1]})

# Spatial plot
sns.scatterplot(x=spatial_coords['CenterX_global_px'], y=spatial_coords['CenterY_global_px'],
                hue=leiden_clusters, palette=cluster_colors, s=2, edgecolor=None, ax=axes[0])
axes[0].set_title('Leiden Clusters on Spatial Image')
axes[0].set_xlabel('CenterX (px)')
axes[0].set_ylabel('CenterY (px)')
axes[0].legend_.remove()  # Remove individual legend

# UMAP plot
sns.scatterplot(x=umap_coords[:, 0], y=umap_coords[:, 1],
                hue=leiden_clusters, palette=cluster_colors, s=5, edgecolor=None, ax=axes[1])
axes[1].set_title('Leiden Clusters on UMAP')
axes[1].set_xlabel('UMAP1')
axes[1].set_ylabel('UMAP2')
axes[1].legend_.remove()  # Remove individual legend

# Create a single shared legend closer to the plots
handles = [plt.Line2D([0], [0], marker='o', color=cluster_colors[str(cluster)], linestyle='', markersize=8)
           for cluster in sorted_clusters]
labels = [str(cluster) for cluster in sorted_clusters]
fig.legend(handles, labels, title="Leiden Clusters", bbox_to_anchor=(1, 0.5), loc='center left')

plt.tight_layout(rect=[0, 0, 0.85, 1])  # Adjust layout to fit legend closer to the plots
plt.show()

In [ ]:
adata1_filtered.write(os.path.join(dataDir_spatial, "MBbrain6Kbasalslide4_reseg_normalized1.h5ad"), compression="gzip")

In [ ]:
#bgs4 reseg raw
adata1 = sc.read_h5ad("/u/project/cluo/mbaig/cosmx/analyses/20250527_MB_basal_2/processed_data/20260106/MBbrain6Kbasalslide4_reseg_raw1.h5ad")
adata1

In [ ]:
#150 umi cutoff

adata1_filtered = adata1[(adata1.obs['nCount_RNA'] > 150) & (adata1.obs['nCount_RNA'] < 4000)].copy()
adata1_filtered

In [ ]:
#polygon extraction

# processing bgs4
from matplotlib.path import Path

spatial_coords = adata1_filtered.obsm['spatial']
polygon_coords = np.array([
    [5400, 14000],
    [5100, 12000],
    [5000, 11000],
    [5000, 10000],
    [5300, 8000],
    [5500, 7000],
    [5000, 6000],
    [5000, 5000],
    [7200, 1800],
    [8000, 6000],
    [8000, 6500],
    [8300, 8000],
    [9000, 9000],
    [10100, 11000],
    [9800, 13500]
])

# Create mask
polygon_path = Path(polygon_coords)
inside_polygon = polygon_path.contains_points(spatial_coords)
bgs4_lge_mge = adata1_filtered[inside_polygon, :].copy()

In [ ]:
plt.rcParams['figure.figsize']=(10,10)
sns.scatterplot(data=bgs4_lge_mge.obs,x='CenterX_global_px', y='CenterY_global_px', s=1, edgecolor=None)  # s parameter sets dot size, edgecolor=None removes outline
plt.show()

In [ ]:
del adata1_filtered

In [ ]:
import scanpy as sc

# Store raw counts before processing
bgs4_lge_mge.layers["counts"] = bgs4_lge_mge.X.copy()

# Normalize and log-transform
sc.pp.normalize_total(bgs4_lge_mge, target_sum=1e4)  # Normalize to 10,000 counts per cell
sc.pp.log1p(bgs4_lge_mge)

# Identify highly variable genes (HVGs)
sc.pp.highly_variable_genes(bgs4_lge_mge, flavor="seurat_v3", n_top_genes=1000)

# Create a new column to mark HVGs
bgs4_lge_mge.var["is_hvg"] = bgs4_lge_mge.var["highly_variable"]

# Do NOT subset adata2 to HVGs, so the full gene set is retained
# Instead, perform PCA and clustering using only the HVGs by specifying the 'use_highly_variable' flag

# Scale data (you can choose to scale all genes or just the HVGs; here we use all genes)
sc.pp.scale(bgs4_lge_mge)

# PCA using only HVGs; this will use the 'highly_variable' annotation
sc.tl.pca(bgs4_lge_mge, svd_solver='arpack', use_highly_variable=True)

# Compute neighbors, UMAP, and Leiden clustering based on the PCA computed using HVGs
sc.pp.neighbors(bgs4_lge_mge)
sc.tl.umap(bgs4_lge_mge, min_dist=0.3, spread=1.2)
sc.tl.leiden(bgs4_lge_mge)

In [ ]:
#now plotting back on spatial image

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Extract spatial and UMAP coordinates
spatial_coords = bgs4_lge_mge.obs[['CenterX_global_px', 'CenterY_global_px']]
umap_coords = bgs4_lge_mge.obsm['X_umap']
leiden_clusters = bgs4_lge_mge.obs['leiden'].astype(str)  # Convert Leiden clusters to string for mapping

# Convert Leiden clusters to integers for proper numerical sorting
leiden_clusters_int = leiden_clusters.astype(int)
sorted_clusters = np.sort(leiden_clusters_int.unique())  # Sort numerically

# Define a consistent color palette
palette = sns.color_palette('tab20', len(sorted_clusters))
cluster_colors = {str(cluster): palette[i] for i, cluster in enumerate(sorted_clusters)}

# Create figure and subplots
fig, axes = plt.subplots(1, 2, figsize=(18, 9), gridspec_kw={'width_ratios': [1, 1]})

# Spatial plot
sns.scatterplot(x=spatial_coords['CenterX_global_px'], y=spatial_coords['CenterY_global_px'],
                hue=leiden_clusters, palette=cluster_colors, s=2, edgecolor=None, ax=axes[0])
axes[0].set_title('Leiden Clusters on Spatial Image')
axes[0].set_xlabel('CenterX (px)')
axes[0].set_ylabel('CenterY (px)')
axes[0].legend_.remove()  # Remove individual legend

# UMAP plot
sns.scatterplot(x=umap_coords[:, 0], y=umap_coords[:, 1],
                hue=leiden_clusters, palette=cluster_colors, s=5, edgecolor=None, ax=axes[1])
axes[1].set_title('Leiden Clusters on UMAP')
axes[1].set_xlabel('UMAP1')
axes[1].set_ylabel('UMAP2')
axes[1].legend_.remove()  # Remove individual legend

# Create a single shared legend closer to the plots
handles = [plt.Line2D([0], [0], marker='o', color=cluster_colors[str(cluster)], linestyle='', markersize=8)
           for cluster in sorted_clusters]
labels = [str(cluster) for cluster in sorted_clusters]
fig.legend(handles, labels, title="Leiden Clusters", bbox_to_anchor=(1, 0.5), loc='center left')

plt.tight_layout(rect=[0, 0, 0.85, 1])  # Adjust layout to fit legend closer to the plots
plt.show()

In [ ]:
bgs4_lge_mge.write(os.path.join(dataDir_spatial, "MBbrain6Kbasalslide4_reseg_poly_normalized2.h5ad"), compression="gzip")

In [ ]:
#re-polygon extracting and saving this time, to here:
# /u/project/cluo/mbaig/cosmx/analyses/20250527_MB_basal_2/processed_data/20251219/MBbrain6Kbasalslide4_striatum_ge.h5ad

In [ ]:
bgs4 = sc.read_h5ad("/u/project/cluo/mbaig/cosmx/analyses/20250527_MB_basal_2/processed_data/20250527/MBbrain6Kbasalslide4redo_unfiltered_normalized1.h5ad")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.path import Path

# STEP 1: Visualize to identify coordinates

fig, ax = plt.subplots(figsize=(12, 12))
spatial_coords = bgs4.obsm['spatial']

# Option A: Color by a marker gene to help identify LGE/MGE/striatum
if 'DL3fg32gX1' in bgs4.var_names:
    gene_expr = bgs4[:, 'DLX1'].X.toarray().flatten() if hasattr(bgs4.X, 'toarray') else bgs4[:, 'DLX1'].X.flatten()
    scatter = ax.scatter(
        spatial_coords[:, 0],
        spatial_coords[:, 1],
        c=gene_expr,
        s=1,
        cmap='magma',
        alpha=0.8
    )
    plt.colorbar(scatter, ax=ax, label='DLX1')
# Option B: Color by leiden clusters if available
elif 'leiden' in bgs4.obs.columns:
    import matplotlib.cm as cm
    clusters = bgs4.obs['leiden'].astype('category')
    colors = cm.tab20(np.linspace(0, 1, len(clusters.cat.categories)))
    color_map = dict(zip(clusters.cat.categories, colors))
    point_colors = [color_map[c] for c in clusters]
    ax.scatter(spatial_coords[:, 0], spatial_coords[:, 1], c=point_colors, s=3, alpha=0.8)
# Option C: Just gray points
else:
    ax.scatter(spatial_coords[:, 0], spatial_coords[:, 1], s=3, alpha=0.5, c='gray')

ax.set_aspect('equal')
ax.set_xlabel('X coordinate')
ax.set_ylabel('Y coordinate')
ax.set_title('Identify LGE/MGE/Striatum region - note the coordinates')
ax.grid(True, alpha=0.3)

# Print coordinate ranges to help
print(f"X range: {spatial_coords[:, 0].min():.0f} to {spatial_coords[:, 0].max():.0f}")
print(f"Y range: {spatial_coords[:, 1].min():.0f} to {spatial_coords[:, 1].max():.0f}")

plt.show()

In [ ]:
# STEP 4: Create subset

bgs4_striatum_ge = bgs4[inside_polygon, :].copy()

print(f"\nCreated bgs4_striatum_ge object:")
print(f"  Cells: {bgs4_striatum_ge.n_obs}")
print(f"  Genes: {bgs4_striatum_ge.n_vars}")

In [ ]:
bgs4_striatum_ge

In [ ]:
#rerunning embedding

bgs4_striatum_ge.layers["counts"] = bgs4_striatum_ge.X.copy()
#sc.pp.normalize_total(bgs4_striatum_ge, inplace=True)
#sc.pp.log1p(bgs4_striatum_ge)
sc.pp.pca(bgs4_striatum_ge)
sc.pp.neighbors(bgs4_striatum_ge)
sc.tl.umap(bgs4_striatum_ge)
sc.tl.leiden(bgs4_striatum_ge)

In [ ]:
#after rerunning leiden and changed code to stretch tissue

import matplotlib.pyplot as plt
import numpy as np
import matplotlib.cm as cm

# Get spatial coordinates and UMAP
spatial_coords = bgs4_striatum_ge.obsm['spatial']
umap_coords = bgs4_striatum_ge.obsm['X_umap']

# Get unique clusters
clusters = sorted(bgs4_striatum_ge.obs['leiden'].unique(), key=lambda x: int(x))

# Generate colors for each cluster (same as scanpy uses)
n_clusters = len(clusters)
colors = cm.tab20(np.linspace(0, 1, n_clusters))
color_map = dict(zip(clusters, colors))

print(f"Plotting {len(clusters)} clusters individually...")

# Plot each cluster one by one
for cluster in clusters:
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))

    # Create mask
    is_cluster = bgs4_striatum_ge.obs['leiden'] == cluster

    # LEFT: Spatial plot
    axes[0].scatter(
        spatial_coords[:, 0],
        spatial_coords[:, 1],
        c='lightgray',
        s=3,
        alpha=0.2
    )
    axes[0].scatter(
        spatial_coords[is_cluster, 0],
        spatial_coords[is_cluster, 1],
        c=[color_map[cluster]],
        s=10,
        alpha=0.9,
        edgecolors='black',
        linewidths=0.3
    )

    # Stretch the spatial plot horizontally to reduce side whitespace
    x_range = spatial_coords[:, 0].max() - spatial_coords[:, 0].min()
    y_range = spatial_coords[:, 1].max() - spatial_coords[:, 1].min()
    stretch_factor = 1.2  # Increase this (e.g., 1.5 or 2.0) to stretch more horizontally #################
    axes[0].set_aspect((x_range / y_range) * stretch_factor)

    axes[0].set_title(f'Cluster {cluster} - Spatial ({is_cluster.sum()} cells)',
                     fontsize=14, fontweight='bold')
    axes[0].set_xlabel('X coordinate')
    axes[0].set_ylabel('Y coordinate')

    # RIGHT: UMAP plot
    axes[1].scatter(
        umap_coords[:, 0],
        umap_coords[:, 1],
        c='lightgray',
        s=10,
        alpha=0.2
    )
    axes[1].scatter(
        umap_coords[is_cluster, 0],
        umap_coords[is_cluster, 1],
        c=[color_map[cluster]],
        s=20,
        alpha=0.9,
        edgecolors='black',
        linewidths=0.3
    )
    axes[1].set_title(f'Cluster {cluster} - UMAP', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('UMAP 1')
    axes[1].set_ylabel('UMAP 2')

    plt.tight_layout()
    plt.show()

In [ ]:
import scanpy as sc

# 1. PCA + neighbors (only if not already done)
#sc.pp.pca(bgs4_striatum_ge, n_comps=50, svd_solver='arpack')
#sc.pp.neighbors(bgs4_striatum_ge, n_neighbors=30, n_pcs=40)

# 2. Multi-resolution Leiden - correct syntax for Scanpy ≥1.9.6 + leidenalg ≥0.10
resolutions = [2, 4]

for res in resolutions:
    print(f"Running Leiden at resolution {res} ...")

    sc.tl.leiden(
        bgs4_striatum_ge,
        resolution=res,
        key_added=f"leiden_res_{res}",      #  leiden_res_0.25, leiden_res_0.5, etc.
        random_state=42,                    # for perfect reproducibility
        n_iterations=2                      # 2 is almost always enough
        # flavor and directed are now ignored - the default is igraph + undirected
    )

    n_clusters = bgs4_striatum_ge.obs[f"leiden_res_{res}"].nunique()
    print(f"    {n_clusters} clusters")

# 3. Make resolution 1.0 the default column
#bgs4_striatum_ge.obs['leiden'] = bgs4_striatum_ge.obs['leiden_res_1.0'].astype('category')

# 4. Confirm success
print("\nSuccess! These columns are now in bgs4_striatum_ge.obs:")
print([c for c in bgs4_striatum_ge.obs.columns if c.startswith('leiden_res_')])
print("Default column: 'leiden'")

# Quick test plot
sc.pl.umap(bgs4_striatum_ge, color='leiden_res_2', legend_loc='on data', size=3)

In [ ]:
#leiden res 2

#after rerunning leiden and changed code to stretch tissue

import matplotlib.pyplot as plt
import numpy as np
import matplotlib.cm as cm

# Get spatial coordinates and UMAP
spatial_coords = bgs4_striatum_ge.obsm['spatial']
umap_coords = bgs4_striatum_ge.obsm['X_umap']

# Get unique clusters
clusters = sorted(bgs4_striatum_ge.obs['leiden_res_2'].unique(), key=lambda x: int(x))

# Generate colors for each cluster (same as scanpy uses)
n_clusters = len(clusters)
colors = cm.tab20(np.linspace(0, 1, n_clusters))
color_map = dict(zip(clusters, colors))

print(f"Plotting {len(clusters)} clusters individually...")

# Plot each cluster one by one
for cluster in clusters:
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))

    # Create mask
    is_cluster = bgs4_striatum_ge.obs['leiden_res_2'] == cluster

    # LEFT: Spatial plot
    axes[0].scatter(
        spatial_coords[:, 0],
        spatial_coords[:, 1],
        c='lightgray',
        s=3,
        alpha=0.2
    )
    axes[0].scatter(
        spatial_coords[is_cluster, 0],
        spatial_coords[is_cluster, 1],
        c=[color_map[cluster]],
        s=10,
        alpha=0.9,
        edgecolors='black',
        linewidths=0.3
    )

    # Stretch the spatial plot horizontally to reduce side whitespace
    x_range = spatial_coords[:, 0].max() - spatial_coords[:, 0].min()
    y_range = spatial_coords[:, 1].max() - spatial_coords[:, 1].min()
    stretch_factor = 1.2  # Increase this (e.g., 1.5 or 2.0) to stretch more horizontally #################
    axes[0].set_aspect((x_range / y_range) * stretch_factor)

    axes[0].set_title(f'Cluster {cluster} - Spatial ({is_cluster.sum()} cells)',
                     fontsize=14, fontweight='bold')
    axes[0].set_xlabel('X coordinate')
    axes[0].set_ylabel('Y coordinate')

    # RIGHT: UMAP plot
    axes[1].scatter(
        umap_coords[:, 0],
        umap_coords[:, 1],
        c='lightgray',
        s=10,
        alpha=0.2
    )
    axes[1].scatter(
        umap_coords[is_cluster, 0],
        umap_coords[is_cluster, 1],
        c=[color_map[cluster]],
        s=20,
        alpha=0.9,
        edgecolors='black',
        linewidths=0.3
    )
    axes[1].set_title(f'Cluster {cluster} - UMAP', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('UMAP 1')
    axes[1].set_ylabel('UMAP 2')

    plt.tight_layout()
    plt.show()

In [ ]:
import scanpy as sc

# 1. PCA + neighbors (only if not already done)
#sc.pp.pca(bgs4_striatum_ge, n_comps=50, svd_solver='arpack')
#sc.pp.neighbors(bgs4_striatum_ge, n_neighbors=30, n_pcs=40)

# 2. Multi-resolution Leiden - correct syntax for Scanpy ≥1.9.6 + leidenalg ≥0.10
resolutions = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9]

for res in resolutions:
    print(f"Running Leiden at resolution {res} ...")

    sc.tl.leiden(
        bgs4_striatum_ge,
        resolution=res,
        key_added=f"leiden_res_{res}",      #  leiden_res_0.25, leiden_res_0.5, etc.
        random_state=42,                    # for perfect reproducibility
        n_iterations=2                      # 2 is almost always enough
        # flavor and directed are now ignored - the default is igraph + undirected
    )

    n_clusters = bgs4_striatum_ge.obs[f"leiden_res_{res}"].nunique()
    print(f"    {n_clusters} clusters")

# 3. Make resolution 1.0 the default column
#bgs4_striatum_ge.obs['leiden'] = bgs4_striatum_ge.obs['leiden_res_1.0'].astype('category')

# 4. Confirm success
print("\nSuccess! These columns are now in bgs4_striatum_ge.obs:")
print([c for c in bgs4_striatum_ge.obs.columns if c.startswith('leiden_res_')])
print("Default column: 'leiden'")

# Quick test plot
#sc.pl.umap(bgs4_striatum_ge, color='leiden_res_2', legend_loc='on data', size=3)

In [ ]:
#leiden res 0.1

#after rerunning leiden and changed code to stretch tissue

import matplotlib.pyplot as plt
import numpy as np
import matplotlib.cm as cm

# Get spatial coordinates and UMAP
spatial_coords = bgs4_striatum_ge.obsm['spatial']
umap_coords = bgs4_striatum_ge.obsm['X_umap']

# Get unique clusters
clusters = sorted(bgs4_striatum_ge.obs['leiden_res_0.1'].unique(), key=lambda x: int(x))

# Generate colors for each cluster (same as scanpy uses)
n_clusters = len(clusters)
colors = cm.tab20(np.linspace(0, 1, n_clusters))
color_map = dict(zip(clusters, colors))

print(f"Plotting {len(clusters)} clusters individually...")

# Plot each cluster one by one
for cluster in clusters:
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))

    # Create mask
    is_cluster = bgs4_striatum_ge.obs['leiden_res_0.1'] == cluster

    # LEFT: Spatial plot
    axes[0].scatter(
        spatial_coords[:, 0],
        spatial_coords[:, 1],
        c='lightgray',
        s=3,
        alpha=0.2
    )
    axes[0].scatter(
        spatial_coords[is_cluster, 0],
        spatial_coords[is_cluster, 1],
        c=[color_map[cluster]],
        s=10,
        alpha=0.9,
        edgecolors='black',
        linewidths=0.3
    )

    # Stretch the spatial plot horizontally to reduce side whitespace
    x_range = spatial_coords[:, 0].max() - spatial_coords[:, 0].min()
    y_range = spatial_coords[:, 1].max() - spatial_coords[:, 1].min()
    stretch_factor = 1.2  # Increase this (e.g., 1.5 or 2.0) to stretch more horizontally #################
    axes[0].set_aspect((x_range / y_range) * stretch_factor)

    axes[0].set_title(f'Cluster {cluster} - Spatial ({is_cluster.sum()} cells)',
                     fontsize=14, fontweight='bold')
    axes[0].set_xlabel('X coordinate')
    axes[0].set_ylabel('Y coordinate')

    # RIGHT: UMAP plot
    axes[1].scatter(
        umap_coords[:, 0],
        umap_coords[:, 1],
        c='lightgray',
        s=10,
        alpha=0.2
    )
    axes[1].scatter(
        umap_coords[is_cluster, 0],
        umap_coords[is_cluster, 1],
        c=[color_map[cluster]],
        s=20,
        alpha=0.9,
        edgecolors='black',
        linewidths=0.3
    )
    axes[1].set_title(f'Cluster {cluster} - UMAP', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('UMAP 1')
    axes[1].set_ylabel('UMAP 2')

    plt.tight_layout()
    plt.show()

In [ ]:
#leiden res 0.4

#after rerunning leiden and changed code to stretch tissue

import matplotlib.pyplot as plt
import numpy as np
import matplotlib.cm as cm

# Get spatial coordinates and UMAP
spatial_coords = bgs4_striatum_ge.obsm['spatial']
umap_coords = bgs4_striatum_ge.obsm['X_umap']

# Get unique clusters
clusters = sorted(bgs4_striatum_ge.obs['leiden_res_0.4'].unique(), key=lambda x: int(x))

# Generate colors for each cluster (same as scanpy uses)
n_clusters = len(clusters)
colors = cm.tab20(np.linspace(0, 1, n_clusters))
color_map = dict(zip(clusters, colors))

print(f"Plotting {len(clusters)} clusters individually...")

# Plot each cluster one by one
for cluster in clusters:
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))

    # Create mask
    is_cluster = bgs4_striatum_ge.obs['leiden_res_0.4'] == cluster

    # LEFT: Spatial plot
    axes[0].scatter(
        spatial_coords[:, 0],
        spatial_coords[:, 1],
        c='lightgray',
        s=3,
        alpha=0.2
    )
    axes[0].scatter(
        spatial_coords[is_cluster, 0],
        spatial_coords[is_cluster, 1],
        c=[color_map[cluster]],
        s=10,
        alpha=0.9,
        edgecolors='black',
        linewidths=0.3
    )

    # Stretch the spatial plot horizontally to reduce side whitespace
    x_range = spatial_coords[:, 0].max() - spatial_coords[:, 0].min()
    y_range = spatial_coords[:, 1].max() - spatial_coords[:, 1].min()
    stretch_factor = 1.2  # Increase this (e.g., 1.5 or 2.0) to stretch more horizontally #################
    axes[0].set_aspect((x_range / y_range) * stretch_factor)

    axes[0].set_title(f'Cluster {cluster} - Spatial ({is_cluster.sum()} cells)',
                     fontsize=14, fontweight='bold')
    axes[0].set_xlabel('X coordinate')
    axes[0].set_ylabel('Y coordinate')

    # RIGHT: UMAP plot
    axes[1].scatter(
        umap_coords[:, 0],
        umap_coords[:, 1],
        c='lightgray',
        s=10,
        alpha=0.2
    )
    axes[1].scatter(
        umap_coords[is_cluster, 0],
        umap_coords[is_cluster, 1],
        c=[color_map[cluster]],
        s=20,
        alpha=0.9,
        edgecolors='black',
        linewidths=0.3
    )
    axes[1].set_title(f'Cluster {cluster} - UMAP', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('UMAP 1')
    axes[1].set_ylabel('UMAP 2')

    plt.tight_layout()
    plt.show()

In [ ]:
#leiden res 0.6

#after rerunning leiden and changed code to stretch tissue

import matplotlib.pyplot as plt
import numpy as np
import matplotlib.cm as cm

# Get spatial coordinates and UMAP
spatial_coords = bgs4_striatum_ge.obsm['spatial']
umap_coords = bgs4_striatum_ge.obsm['X_umap']

# Get unique clusters
clusters = sorted(bgs4_striatum_ge.obs['leiden_res_0.6'].unique(), key=lambda x: int(x))

# Generate colors for each cluster (same as scanpy uses)
n_clusters = len(clusters)
colors = cm.tab20(np.linspace(0, 1, n_clusters))
color_map = dict(zip(clusters, colors))

print(f"Plotting {len(clusters)} clusters individually...")

# Plot each cluster one by one
for cluster in clusters:
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))

    # Create mask
    is_cluster = bgs4_striatum_ge.obs['leiden_res_0.6'] == cluster

    # LEFT: Spatial plot
    axes[0].scatter(
        spatial_coords[:, 0],
        spatial_coords[:, 1],
        c='lightgray',
        s=3,
        alpha=0.2
    )
    axes[0].scatter(
        spatial_coords[is_cluster, 0],
        spatial_coords[is_cluster, 1],
        c=[color_map[cluster]],
        s=10,
        alpha=0.9,
        edgecolors='black',
        linewidths=0.3
    )

    # Stretch the spatial plot horizontally to reduce side whitespace
    x_range = spatial_coords[:, 0].max() - spatial_coords[:, 0].min()
    y_range = spatial_coords[:, 1].max() - spatial_coords[:, 1].min()
    stretch_factor = 1.2  # Increase this (e.g., 1.5 or 2.0) to stretch more horizontally #################
    axes[0].set_aspect((x_range / y_range) * stretch_factor)

    axes[0].set_title(f'Cluster {cluster} - Spatial ({is_cluster.sum()} cells)',
                     fontsize=14, fontweight='bold')
    axes[0].set_xlabel('X coordinate')
    axes[0].set_ylabel('Y coordinate')

    # RIGHT: UMAP plot
    axes[1].scatter(
        umap_coords[:, 0],
        umap_coords[:, 1],
        c='lightgray',
        s=10,
        alpha=0.2
    )
    axes[1].scatter(
        umap_coords[is_cluster, 0],
        umap_coords[is_cluster, 1],
        c=[color_map[cluster]],
        s=20,
        alpha=0.9,
        edgecolors='black',
        linewidths=0.3
    )
    axes[1].set_title(f'Cluster {cluster} - UMAP', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('UMAP 1')
    axes[1].set_ylabel('UMAP 2')

    plt.tight_layout()
    plt.show()

In [ ]:
#leiden res 0.8

#after rerunning leiden and changed code to stretch tissue

import matplotlib.pyplot as plt
import numpy as np
import matplotlib.cm as cm

# Get spatial coordinates and UMAP
spatial_coords = bgs4_striatum_ge.obsm['spatial']
umap_coords = bgs4_striatum_ge.obsm['X_umap']

# Get unique clusters
clusters = sorted(bgs4_striatum_ge.obs['leiden_res_0.8'].unique(), key=lambda x: int(x))

# Generate colors for each cluster (same as scanpy uses)
n_clusters = len(clusters)
colors = cm.tab20(np.linspace(0, 1, n_clusters))
color_map = dict(zip(clusters, colors))

print(f"Plotting {len(clusters)} clusters individually...")

# Plot each cluster one by one
for cluster in clusters:
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))

    # Create mask
    is_cluster = bgs4_striatum_ge.obs['leiden_res_0.8'] == cluster

    # LEFT: Spatial plot
    axes[0].scatter(
        spatial_coords[:, 0],
        spatial_coords[:, 1],
        c='lightgray',
        s=3,
        alpha=0.2
    )
    axes[0].scatter(
        spatial_coords[is_cluster, 0],
        spatial_coords[is_cluster, 1],
        c=[color_map[cluster]],
        s=10,
        alpha=0.9,
        edgecolors='black',
        linewidths=0.3
    )

    # Stretch the spatial plot horizontally to reduce side whitespace
    x_range = spatial_coords[:, 0].max() - spatial_coords[:, 0].min()
    y_range = spatial_coords[:, 1].max() - spatial_coords[:, 1].min()
    stretch_factor = 1.2  # Increase this (e.g., 1.5 or 2.0) to stretch more horizontally #################
    axes[0].set_aspect((x_range / y_range) * stretch_factor)

    axes[0].set_title(f'Cluster {cluster} - Spatial ({is_cluster.sum()} cells)',
                     fontsize=14, fontweight='bold')
    axes[0].set_xlabel('X coordinate')
    axes[0].set_ylabel('Y coordinate')

    # RIGHT: UMAP plot
    axes[1].scatter(
        umap_coords[:, 0],
        umap_coords[:, 1],
        c='lightgray',
        s=10,
        alpha=0.2
    )
    axes[1].scatter(
        umap_coords[is_cluster, 0],
        umap_coords[is_cluster, 1],
        c=[color_map[cluster]],
        s=20,
        alpha=0.9,
        edgecolors='black',
        linewidths=0.3
    )
    axes[1].set_title(f'Cluster {cluster} - UMAP', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('UMAP 1')
    axes[1].set_ylabel('UMAP 2')

    plt.tight_layout()
    plt.show()

In [ ]:
#leiden res 1.4

#after rerunning leiden and changed code to stretch tissue

import matplotlib.pyplot as plt
import numpy as np
import matplotlib.cm as cm

# Get spatial coordinates and UMAP
spatial_coords = bgs4_striatum_ge.obsm['spatial']
umap_coords = bgs4_striatum_ge.obsm['X_umap']

# Get unique clusters
clusters = sorted(bgs4_striatum_ge.obs['leiden_res_1.4'].unique(), key=lambda x: int(x))

# Generate colors for each cluster (same as scanpy uses)
n_clusters = len(clusters)
colors = cm.tab20(np.linspace(0, 1, n_clusters))
color_map = dict(zip(clusters, colors))

print(f"Plotting {len(clusters)} clusters individually...")

# Plot each cluster one by one
for cluster in clusters:
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))

    # Create mask
    is_cluster = bgs4_striatum_ge.obs['leiden_res_1.4'] == cluster

    # LEFT: Spatial plot
    axes[0].scatter(
        spatial_coords[:, 0],
        spatial_coords[:, 1],
        c='lightgray',
        s=3,
        alpha=0.2
    )
    axes[0].scatter(
        spatial_coords[is_cluster, 0],
        spatial_coords[is_cluster, 1],
        c=[color_map[cluster]],
        s=10,
        alpha=0.9,
        edgecolors='black',
        linewidths=0.3
    )

    # Stretch the spatial plot horizontally to reduce side whitespace
    x_range = spatial_coords[:, 0].max() - spatial_coords[:, 0].min()
    y_range = spatial_coords[:, 1].max() - spatial_coords[:, 1].min()
    stretch_factor = 1.2  # Increase this (e.g., 1.5 or 2.0) to stretch more horizontally #################
    axes[0].set_aspect((x_range / y_range) * stretch_factor)

    axes[0].set_title(f'Cluster {cluster} - Spatial ({is_cluster.sum()} cells)',
                     fontsize=14, fontweight='bold')
    axes[0].set_xlabel('X coordinate')
    axes[0].set_ylabel('Y coordinate')

    # RIGHT: UMAP plot
    axes[1].scatter(
        umap_coords[:, 0],
        umap_coords[:, 1],
        c='lightgray',
        s=10,
        alpha=0.2
    )
    axes[1].scatter(
        umap_coords[is_cluster, 0],
        umap_coords[is_cluster, 1],
        c=[color_map[cluster]],
        s=20,
        alpha=0.9,
        edgecolors='black',
        linewidths=0.3
    )
    axes[1].set_title(f'Cluster {cluster} - UMAP', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('UMAP 1')
    axes[1].set_ylabel('UMAP 2')

    plt.tight_layout()
    plt.show()

In [ ]:
dataDir_spatial_20251219 = '/u/home/m/mbaig/project-cluo/cosmx/analyses/20250527_MB_basal_2/processed_data/20251219/'

In [ ]:
bgs4_striatum_ge.write(os.path.join(dataDir_spatial_20251219, "MBbrain6Kbasalslide4_striatum_ge.h5ad"), compression="gzip")